# QuantumCircuit.jl Phase 5 Circuit Capacitive Coupling

**Audience**
- Julia users who already know the baseline `CompositeSystem`, spectrum, and dynamics APIs and want the exact coupled circuit-mode path.

**Prerequisites**
- Basic Julia syntax.
- Familiarity with `CircuitHamiltonianSpec`, `spectrum`, `simulate_sweep`, and `evolve`.

**What you will learn**
- How `CircuitCapacitiveCoupling(...; G = ...)` differs from the older effective `CapacitiveCoupling(...; g = ...)` path.
- How to build the supported exact circuit-mode pair families from Lagemann Eqs. (5) and (6).
- How to run `:G` sweeps, coupled local-drive dynamics, and coupled `FluxControl` in exact circuit mode.


## Outline

1. Activate the local package environment.
2. Review the current exact circuit-mode support matrix.
3. Build a transmon-like↔transmon-like exact circuit Hamiltonian.
4. Build a resonator↔transmon-like exact circuit Hamiltonian.
5. Run a `:G` sweep with the generic sweep API.
6. Run coupled local-drive dynamics in exact circuit mode.
7. Run a coupled exact circuit-mode `FluxControl` example.
8. Review the current limits.


In [2]:
using Pkg

function find_repo_root(start::AbstractString)
    candidates = [
        normpath(start),
        normpath(joinpath(start, "..")),
        normpath(joinpath(start, "..", "..")),
    ]

    for candidate in unique(candidates)
        project_toml = joinpath(candidate, "Project.toml")
        if isfile(project_toml)
            content = read(project_toml, String)
            occursin("QuantumCircuit", content) && return candidate
        end
    end

    error("Could not find the QuantumCircuit.jl project root. Start Jupyter from the repository or open the notebook from inside it.")
end

project_root = find_repo_root(pwd())
Pkg.activate(project_root)
Pkg.instantiate()

using QuantumCircuit
using UnicodePlots: lineplot, lineplot!


  Activating project at `~/Research/20_Projects/QuantumCircuit.jl`




## Step 1 - Exact circuit-mode support matrix

This notebook stays inside the currently supported exact charge-basis surface. The interaction forms follow the paper’s circuit Hamiltonians: Eq. (5) for transmon-like↔transmon-like `G n_i n_j` and Eq. (6) for resonator↔transmon-like `G (a + a^†) n`.

| Workflow | Supported here? | Notes |
| --- | --- | --- |
| `CircuitCapacitiveCoupling(...; G = ...)` with transmon-like↔transmon-like | Yes | Exact charge-charge interaction. |
| `CircuitCapacitiveCoupling(...; G = ...)` with resonator↔transmon-like | Yes | Exact resonator-quadrature to charge interaction. |
| `SweepSpec(..., :G, ...)` under `CircuitHamiltonianSpec` | Yes | Reuses the generic sweep surface. |
| Local `SubsystemDrive` dynamics in exact circuit mode | Yes | Coupled exact circuit systems are supported for the two pair families above. |
| Coupled `FluxControl` in exact circuit mode | Yes | The coupling term stays static; tunable local terms carry the flux dependence. |
| `CapacitiveCoupling(...; g = ...)` in circuit mode | No | That remains the effective exchange coupling surface. |
| Resonator↔resonator exact circuit coupling | No | Still out of scope in this milestone. |
| Paper-derived effective `g(t)` and `ḡ(t)` | No | Deferred to a later effective-model fidelity pass. |


## Step 2 - Transmon-like↔transmon-like exact circuit coupling

The exact circuit interaction uses the bare embedded charge operators, not the offset-charge term `(n - ng)`. This is the direct charge-charge form from Eq. (5).


In [3]:
circuit_spec = CircuitHamiltonianSpec(charge_cutoff = 3)

q1 = TunableTransmon(:q1; EJmax = 20.0, EC = 0.25, flux = 0.12, asymmetry = 0.20, ng = 0.05, ncut = 6)
q2 = TunableCoupler(:q2; EJmax = 15.0, EC = 0.30, flux = 0.08, asymmetry = 0.10, ng = -0.03, ncut = 5)
qq_sys = CompositeSystem(q1, q2, CircuitCapacitiveCoupling(:q1, :q2; G = 0.035))

qq_model = build_model(qq_sys; hamiltonian_spec = circuit_spec)
qq_spec = spectrum(qq_sys; levels = 5, hamiltonian_spec = circuit_spec)
qq_charge_charge = charge_operator(qq_model, :q1) * charge_operator(qq_model, :q2)

(; dimensions = qq_model.dimensions,
   transition_01 = transition_frequencies(qq_spec)[1],
   anharmonicity = anharmonicity(qq_spec),
   charge_charge_size = size(qq_charge_charge.data))


(dimensions = Dict(:q2 => 7, :q1 => 7), transition_01 = 5.773026349895915, anharmonicity = -5.208700652108128, charge_charge_size = (49, 49))

## Step 3 - Resonator↔transmon-like exact circuit coupling

For the resonator-transmon pair, the interaction uses the resonator `x` quadrature `(a + a^†)` times the transmon-like bare charge operator, following Eq. (6).


In [4]:
q = Transmon(:q; EJ = 20.0, EC = 0.25, ng = 0.05, ncut = 5)
r = Resonator(:r; ω = 6.2, dim = 3)
rq_sys = CompositeSystem(q, r, CircuitCapacitiveCoupling(:q, :r; G = 0.03))

rq_model = build_model(rq_sys; hamiltonian_spec = circuit_spec)
rq_spec = spectrum(rq_sys; levels = 5, hamiltonian_spec = circuit_spec)
rq_interaction = quadrature_operator(rq_model, :r, :x) * charge_operator(rq_model, :q)

(; dimensions = rq_model.dimensions,
   transition_01 = transition_frequencies(rq_spec)[1],
   lowest_energies = rq_spec.energies[1:4],
   interaction_size = size(rq_interaction.data))


(dimensions = Dict(:q => 7, :r => 3), transition_01 = 6.197138555319448, lowest_energies = [-16.806047127732146, -10.608908572412698, -10.16529553904608, -4.4114680108082975], interaction_size = (21, 21))

## Step 4 - Sweep the exact circuit coupling strength

The generic sweep API can target the circuit coupling parameter `:G` without introducing a separate coupled-circuit sweep interface.


In [5]:
G_values = collect(0.01:0.01:0.05)
G_sweep = SweepSpec(:q, :r, :G, G_values; levels = 4)
G_result = simulate_sweep(rq_sys, G_sweep; hamiltonian_spec = circuit_spec)
G_curve = transition_curve(G_result)
G_summary = sweep_summary(G_result)

lineplot(G_summary.values, G_curve.data;
         xlabel = "G",
         ylabel = "ω01",
         title = "Exact circuit-mode transition under a :G sweep",
         name = "ω01")


             Exact circuit-mode transition under a :G sweep    
             ┌────────────────────────────────────────┐    
         6.2 │⠤⣀⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀│ ω01
             │⠀⠀⠈⠉⠒⠢⢄⣀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀│    
             │⠀⠀⠀⠀⠀⠀⠀⠀⠉⠑⠢⢄⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀│    
             │⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠒⠤⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀│    
             │⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠑⠢⣀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀│    
             │⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠙⢄⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀│    
             │⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠢⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀│    
   ω01       │⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠑⢄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀│    
             │⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠑⠢⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀│    
             │⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠒⢆⠀⠀⠀⠀⠀⠀⠀⠀⠀│    
             │⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠑⢄⠀⠀⠀⠀⠀⠀⠀│    
             │⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠑⢄⠀⠀⠀⠀⠀│    
             │⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠑⢄⠀⠀⠀│    
             │⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠑⢄⠀│    
       6.192 │⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀

## Step 5 - Coupled local-drive dynamics in exact circuit mode

This example drives the resonator through `:x` and watches the response transfer into the coupled transmon through the exact circuit interaction.


In [6]:
drive_spec = CircuitHamiltonianSpec(charge_cutoff = 2)
drive_q = Transmon(:q; EJ = 20.0, EC = 0.25, ng = 0.05, ncut = 5)
drive_r = Resonator(:r; ω = 6.2, dim = 3)
drive_sys = CompositeSystem(drive_q, drive_r, CircuitCapacitiveCoupling(:q, :r; G = 0.03))

ψ0_drive = basis_state(drive_sys; hamiltonian_spec = drive_spec, q = 0, r = 0)
tlist_drive = collect(range(0.0, 8.0; length = 161))

resonator_drive = SubsystemDrive(
    :r_drive,
    :r,
    :x,
    (p, t) -> p.Ω * cos(p.ωd * t),
)

observables_drive = [
    ObservableSpec(:charge_q, :q, :charge),
    ObservableSpec(:nr, :r, :n),
]

drive_params = (; Ω = 0.18, ωd = 6.2)

baseline_drive = evolve(
    drive_sys,
    ψ0_drive,
    tlist_drive;
    hamiltonian_spec = drive_spec,
    observables = observables_drive,
)

driven_result = evolve(
    drive_sys,
    ψ0_drive,
    tlist_drive;
    hamiltonian_spec = drive_spec,
    drives = [resonator_drive],
    observables = observables_drive,
    params = drive_params,
)

charge_baseline = observable_trace(baseline_drive, :charge_q).values
charge_driven = observable_trace(driven_result, :charge_q).values
resonator_number = observable_trace(driven_result, :nr).values
q1_population = population_trace(driven_result, :q, 1).values

drive_plot = lineplot(tlist_drive, real.(charge_baseline);
                     xlabel = "t",
                     ylabel = "<charge_q>",
                     title = "Transferred response under a resonator :x drive",
                     name = "baseline")
lineplot!(drive_plot, tlist_drive, real.(charge_driven); name = "driven")

(; charge_plot = drive_plot,
   max_charge_difference = maximum(abs.(charge_driven .- charge_baseline)),
   max_resonator_population = maximum(real.(resonator_number)),
   max_q_level1_population = maximum(q1_population))


(charge_plot =                  Transferred response under a resonator :x drive         
                 ┌────────────────────────────────────────┐         
               2 │⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀│ baseline
                 │⠀⣼⠀⠀⠀⠀⠀⢠⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⣶⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀│ driven  
                 │⠀⡏⡆⠀⠀⠀⠀⣸⡇⠀⠀⠀⠀⢰⡀⠀⠀⡀⠀⠀⠀⠀⠀⣿⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀│         
                 │⠀⡇⡇⠀⢀⠀⠀⡇⡇⠀⠀⠀⠀⡞⡇⠀⢰⣇⠀⢀⡄⠀⢸⢹⠀⠀⠀⠀⠀⣀⠀⠀⠀⠀⠀⠀⠀⠀⢠⡄│         
                 │⠀⡇⡇⠀⡾⡄⠀⡇⡇⠀⠀⠀⠀⡇⡇⠀⢸⢸⠀⢸⡇⠀⢸⠸⡄⠀⠀⠀⠀⣿⠀⠀⠀⠀⠀⠀⠀⠀⡾⡇│         
                 │⠀⡇⡇⠀⡇⡇⠀⡇⢱⠀⢰⡆⠀⡇⡇⠀⢸⢸⠀⢸⢸⠀⢸⠀⡇⠀⣶⠀⢰⠛⡆⠀⢠⡆⠀⣤⠀⠀⡇⣧│         
                 │⢸⠀⡇⠀⡇⡇⠀⡇⢸⠀⡞⢇⠀⡇⢹⠀⡞⢸⠀⡞⢸⠀⢸⠀⡇⠀⡏⡇⢸⠀⡇⠀⡞⢇⢰⢻⠀⠀⡇⢸│         
   <charge_q>    │⢼⠤⢼⢴⠥⡧⢤⠧⢼⠤⡧⢼⢼⠥⢼⠤⡧⠼⡤⡧⢼⠤⣼⠤⡧⢤⠧⡧⢼⠤⡧⠤⡧⠼⠼⠬⡧⢼⠧⢼│         
                 │⢸⠀⢸⢸⠀⢇⢸⠀⢸⠀⡇⢸⢸⠀⢸⠀⡇⠀⡇⡇⢸⠀⡇⠀⡇⢸⠀⢣⡎⠀⢳⢰⠃⠀⠀⠀⡇⢸⠀⠈│         
                 │⢸⠀⢸⢸⠀⢸⢸⠀⢸⣀⡇⢸⣸⠀⢸⠀⡇⠀⣧⠇⠀⡇⡇⠀⢸⢸⠀⠘⠇⠀⢸⢸⠀⠀⠀⠀⢳⢸⠀⠀│         
                 │⢸⠀⢸⣸⠀⢸⢸⠀⠀⣿⠀⠀⠃⠀⢸⡀⡇⠀⠙⠀⠀⡇⡇⠀⢸⡼⠀⠀⠀⠀⢸⣸⠀⠀⠀⠀⢸⡏⠀⠀│         
                 │⣸⠀⠀⠇⠀⢸⣸⠀⠀⣿⠀⠀⠀⠀⠀⣿⠁⠀⠀⠀⠀⡇⡇⠀⢸⡇⠀⠀⠀⠀⠈⠃⠀⠀⠀⠀⠈⠃⠀⠀│         
              

## Step 6 - Coupled exact circuit-mode `FluxControl`

The exact circuit interaction stays static while the tunable subsystem’s local `cosphi` and `sinphi` terms change with flux. This is the coupled exact-circuit flux path, not the paper’s later effective `g(t)` reduction.


In [7]:
flux_spec = CircuitHamiltonianSpec(charge_cutoff = 3)
flux_q = TunableTransmon(:q; EJmax = 20.0, EC = 0.25, flux = 0.12, asymmetry = 0.20, ng = 0.05, ncut = 6)
flux_r = Resonator(:r; ω = 6.0, dim = 2)
flux_sys = CompositeSystem(flux_q, flux_r, CircuitCapacitiveCoupling(:q, :r; G = 0.025))

ψ0_flux = basis_state(flux_sys; hamiltonian_spec = flux_spec, q = 0, r = 0)
tlist_flux = collect(range(0.0, 6.0; length = 121))

flux_pulse = FluxControl(
    :flux_pulse,
    :q,
    (p, t) -> p.δ * sin(p.ω * t);
    derivative = (p, t) -> p.δ * p.ω * cos(p.ω * t),
)

observables_flux = [
    ObservableSpec(:charge_q, :q, :charge),
    ObservableSpec(:nr, :r, :n),
]

pulse_params = (; δ = 0.08, ω = 5.6)

baseline_flux = evolve(
    flux_sys,
    ψ0_flux,
    tlist_flux;
    hamiltonian_spec = flux_spec,
    observables = observables_flux,
)

pulsed_flux = evolve(
    flux_sys,
    ψ0_flux,
    tlist_flux;
    hamiltonian_spec = flux_spec,
    observables = observables_flux,
    flux_controls = [flux_pulse],
    params = pulse_params,
)

charge_flux_baseline = observable_trace(baseline_flux, :charge_q).values
charge_flux_pulsed = observable_trace(pulsed_flux, :charge_q).values
resonator_flux_pulsed = observable_trace(pulsed_flux, :nr).values
flux_level1 = population_trace(pulsed_flux, :q, 1).values

flux_plot = lineplot(tlist_flux, real.(charge_flux_baseline);
                    xlabel = "t",
                    ylabel = "<charge_q>",
                    title = "Coupled exact circuit-mode flux response",
                    name = "baseline")
lineplot!(flux_plot, tlist_flux, real.(charge_flux_pulsed); name = "pulsed")

(; charge_plot = flux_plot,
   max_charge_difference = maximum(abs.(charge_flux_pulsed .- charge_flux_baseline)),
   max_resonator_population = maximum(real.(resonator_flux_pulsed)),
   max_q_level1_population = maximum(flux_level1))


(charge_plot =                  ⠀Coupled exact circuit-mode flux response⠀         
                 ┌────────────────────────────────────────┐         
               3 │⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀│ baseline
                 │⠀⠀⠀⠀⠀⠀⠀⠀⢰⢳⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀│ pulsed  
                 │⠀⠀⠀⠀⠀⠀⠀⠀⣸⢸⡆⠀⠀⠀⢰⢆⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀│         
                 │⠀⠀⢰⣆⠀⠀⠀⠀⡇⠈⡇⠀⠀⠀⣼⢸⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀│         
                 │⠀⠀⣼⢸⡀⠀⠀⢀⡇⠀⣇⠀⠀⠀⡟⠸⡆⠀⠀⠀⢀⢦⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⢀│         
                 │⠀⠀⡏⠘⡇⠀⠀⢸⠀⠀⢹⠀⠀⢰⡇⠀⡇⠀⠀⠀⢸⡸⡀⠀⠀⠀⢠⠛⡄⠀⠀⢀⠔⡩⣆⠀⠀⣀⡰⠃│         
                 │⠀⢠⡇⠀⡇⠀⠀⢸⠀⠀⢸⠀⠀⣸⠇⠀⢳⠀⠀⠀⡼⢱⡇⠀⠀⠀⢸⠒⣇⠀⠀⡎⢠⠃⠘⣧⡜⡜⢇⡀│         
   <charge_q>    │⠤⢼⠥⠤⢿⠤⠤⡾⠤⠤⠼⡦⠤⣿⠤⠤⢼⠤⠤⠤⡧⠼⣧⠤⠤⠤⡯⠤⠼⡦⢤⠧⡮⠤⠤⢵⢤⠧⠤⠼│         
                 │⠀⣸⠀⠀⢸⠀⠀⡇⠀⠀⠀⡇⢠⡟⠀⠀⠈⡇⠀⢰⠃⠀⢻⡄⠀⢰⡇⠀⠀⢷⠎⢰⠁⠀⠀⠈⠋⠀⠀⠀│         
                 │⠀⡟⠀⠀⢸⡆⢠⠇⠀⠀⠀⢱⡜⡇⠀⠀⠀⢧⠀⣼⠀⠀⠈⣇⠀⣼⠁⠀⠀⢸⠀⡜⠀⠀⠀⠀⠀⠀⠀⠀│         
                 │⠀⡇⠀⠀⠀⡇⢸⠀⠀⠀⠀⢸⣱⠃⠀⠀⠀⢸⣤⡏⠀⠀⠀⢻⣄⡿⠀⠀⠀⠈⣦⠇⠀⠀⠀⠀⠀⠀⠀⠀│         
                 │⢰⠃⠀⠀⠀⣧⡜⠀⠀⠀⠀⠀⠋⠀⠀⠀⠀⠀⠫⠃⠀⠀⠀⠘⡿⠇⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀│         
                 │⢸

## Current limits

- `CapacitiveCoupling(...; g = ...)` still belongs to `EffectiveHamiltonianSpec()`; use `CircuitCapacitiveCoupling(...; G = ...)` for exact circuit mode.
- Supported exact circuit-mode pair families are transmon-like↔transmon-like and resonator↔transmon-like.
- Resonator↔resonator exact circuit coupling is still out of scope.
- Paper-derived effective time-dependent couplings such as `g(t)` and `ḡ(t)` are not part of this notebook.
- The uncoupled circuit-mode notebook remains the best entry point for circuit-native operators and single-device charge-basis workflows.
